In [ ]:
suppressPackageStartupMessages({
library(DESeq2)
library(ggplot2)
library(pheatmap)
library(tidyverse)
library(here)
    })

In [ ]:
setwd('..')

In [ ]:
getwd()

In [ ]:
utils <- new.env()

sys.source(here::here("scripts", "utils.r"), envir = utils)

In [ ]:
data_path <- "./Data/Bulk-seq_data/GSE252692_feature_counts/"
files <- list.files(data_path, pattern = "\\.txt$", full.names = TRUE)

In [ ]:
files

In [ ]:
merged_df <- files %>%
  lapply(function(file) {
    data <- read.delim(file, skip = 1)
    sample_name <- gsub("_count_matrix.txt", "", basename(file))
    selected_data <- data %>% dplyr::select(Geneid = 1, last_col())
    colnames(selected_data)[2] <- sample_name
    return(selected_data)
  }) %>%
  purrr::reduce(full_join, by = "Geneid")

In [ ]:
write_csv(merged_df,'./CSV/MRC-5_raw_counts.csv')

In [ ]:
sample_names <- colnames(merged_df)[-1]
condition <- rep(x = c("Control","3hpi","6hpi","9hpi","12hpi","18hpi","24hpi","30hpi"),times = 3)

In [ ]:
count_data <- merged_df[2:25]
rownames(count_data) <- merged_df$Geneid

In [ ]:
count_data

In [ ]:
condition

In [ ]:
sample_names

In [ ]:
coldata <- data.frame(row.names = sample_names, condition = factor(condition))

In [ ]:
dds <- DESeqDataSetFromMatrix(countData = count_data, colData = coldata, design = ~ condition)
dds <- DESeq(dds)

In [ ]:
resultsNames(dds)

In [ ]:
target_timepoints <- c("3hpi", "6hpi", "9hpi", "12hpi", "18hpi", "24hpi", "30hpi")

lapply(target_timepoints, function(tp) {
    utils$analyze_save(dds, 'condition',coefficient = tp, "Control", './CSV/Bulk-seq/MRC-5/', lfc_cutoff = 0.58)
})

In [ ]:
sessionInfo()